# CatBoost flood impact model: walkthrough

This notebook runs the pipeline in `model/impact.py` one step at a time. It does not hold its own copy of the code. Every step calls the function the training script and the tests use, so the numbers here are the ones in `model/artifacts/impact/impact_metrics.json`.

The model is the one in `data/CatBoost_Model_Executive_Brief.pdf`: a CatBoost regressor for the DFO **Flood impact index**, depth 7, learning rate 0.03, L2 5.0, MVS subsample 0.8, up to 1,200 trees.

Two findings shape everything below:

1. The index is a formula, `log10(days × severity × area_km2)`. Given those three numbers there is nothing left to learn.
2. The 14 weather columns in the enriched file do not behave like observations. Treat them as synthetic.

Run from the repo root or from `model/notebooks/`. Needs `pip install -r model/requirements.txt`. Training cells take about 10 seconds each.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "model" / "impact.py").exists())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from model import impact

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
ROOT

PosixPath('/home/dennis/Desktop/Projects/DrainSense-Global')

## 1. Load and clean

`load_records()` reads `data/Enriched_Global_Flood_Records-v2.csv` and does five things:

- strips the thousands separators from the area
- parses `Clean_Start_Date` and `Clean_End_Date`
- adds `Days_Inclusive` (end − start + 1) and `Log_Area`
- maps the free-text `Main Cause` into 10 groups with `CAUSE_RULES`
- keeps the 14 weather columns as they are

The training script writes the result to `model/artifacts/impact/event_features.csv`, and `read_events()` reads it back.

In [2]:
frame = impact.load_records(ROOT / "data" / "Enriched_Global_Flood_Records-v2.csv")
print(frame.shape)
frame[["Event_ID", "Start_Date", "End_Date", "Country", "Area_km2", "Days_Inclusive",
       "Event_Duration_Days", "Severity", "Cause_Group", impact.TARGET]].head()

(5503, 33)


,Event_ID,Start_Date,End_Date,Country,Area_km2,Days_Inclusive,Event_Duration_Days,Severity,Cause_Group,Flood_Impact_Index
0,1,1985-01-01,1985-01-05,Algeria,92616.0,5,4,1.0,heavy_rain,5.7
1,2,1985-01-15,1985-02-02,Brazil,678499.0,19,18,2.0,heavy_rain,7.4
2,3,1985-01-20,1985-01-21,Philippines,12846.0,2,1,1.0,torrential_rain,4.4
3,4,1985-02-04,1985-02-18,Indonesia,16542.0,15,14,1.0,torrential_rain,5.4
4,5,1985-02-09,1985-02-11,Mozambique,20082.0,3,2,2.0,heavy_rain,5.1


In [3]:
print(frame["Main_Cause"].nunique(), "distinct cause strings")
frame["Cause_Group"].value_counts()

211 distinct cause strings


Cause_Group
heavy_rain             3398
torrential_rain         818
tropical_cyclone        507
monsoon                 452
snowmelt_or_ice         159
dam_or_levee             71
other                    60
extratropical_storm      20
surge_or_tide            15
other_storm               3
Name: count, dtype: int64

## 2. The target is a formula

DFO defines flood magnitude as `log10(duration × severity × affected area)`. Duration counts both the start and the end date. Here is that formula set against the column the brief calls the Flood Impact Index:

In [4]:
formula = impact.dfo_magnitude(frame["Days_Inclusive"], frame["Severity"], frame["Area_km2"])
gap = formula - frame[impact.TARGET]
print("RMSE formula vs index:", round(float(np.sqrt((gap ** 2).mean())), 4))
print("RMSE that rounding to 0.1 alone produces:", round(0.1 / np.sqrt(12), 4))
print("rows equal after rounding:", int((np.abs(np.round(formula, 1) - frame[impact.TARGET]) < 0.05).sum()), "of", len(frame))
frame.assign(formula=formula.round(4))[np.abs(np.round(formula, 1) - frame[impact.TARGET]) >= 0.05][
    ["Event_ID", "Area_km2", "Days_Inclusive", "Severity", "formula", impact.TARGET]]

RMSE formula vs index: 0.0288
RMSE that rounding to 0.1 alone produces: 0.0289
rows equal after rounding: 5499 of 5503


,Event_ID,Area_km2,Days_Inclusive,Severity,formula,Flood_Impact_Index
1264,1266,11247.0,5,1.0,4.750,4.7
2849,2852,7029.0,40,2.0,5.750,5.8
5012,5016,22006.0,27,1.5,5.950,6.0
5179,5183,2.0,1,1.0,0.301,0.2


Three of the four misses sit on a rounding boundary, such as 4.75. Event 5183 is a typo in the source, 0.2 where the formula gives 0.30.

The stored `Event_Duration_Days` column is end − start, with same-day events stored as 1. It is not the formula's day count:

In [5]:
stored = impact.dfo_magnitude(frame["Event_Duration_Days"], frame["Severity"], frame["Area_km2"])
print("RMSE with the stored duration column:", round(float(np.sqrt(((stored - frame[impact.TARGET]) ** 2).mean())), 4))
print("same-day events:", int((frame["Days_Inclusive"] == 1).sum()))

RMSE with the stored duration column: 0.1374
same-day events: 196


## 3. Are the weather columns observations?

`data_checks()` collects the numbers. Observed weather at a fixed point would not depend on how large the mapped flood was. Its daily temperature spread would also vary more than this.

In [6]:
pd.Series(impact.data_checks(frame)).drop(["rows_where_it_does_not"]).to_frame("value")

,value
rows,5503
formula,log10(days_inclusive x severity x area_km2)
formula_rmse_vs_index,0.0288
rounding_floor_rmse,0.0289
rows_where_rounded_formula_equals_index,5499
formula_rmse_with_stored_duration_column,0.1374
same_day_events_where_stored_duration_is_1,196
distinct_event_coordinates,376
log_total_precip_r2_from_days_and_abs_latitude,0.7407
log_total_precip_r2_adding_log_area_and_severity,0.8342


Two spot checks against the Open-Meteo archive at the file's own coordinates:

| Event | File | Open-Meteo |
|---|---|---|
| Germany, 14–19 Jul 2021, (52.50, 13.37) | 9.6 °C mean, 274.7 mm | 17.3–23.2 °C daily means, 7.5 mm |
| Kenya, 21–30 Apr 2023, (−1.28, 36.82) | 27.3 °C mean, 223.9 mm | 17.8–20.3 °C daily means, 87.5 mm |

The enrichment step is not in this repo, so the columns stay labelled synthetic.

## 4. Feature sets

- **`brief`** reproduces the brief. With log area, duration, the 14 weather columns, climate zone and continent, the convergence curve matches the brief's: first-iteration test RMSE 1.08 (brief: 1.0963), final train RMSE 0.075 (brief: 0.0695), and area, duration and ET0 at the top of the importance list. Severity is left out. That gap is most of the brief's 0.0985: the standard deviation of log10(severity) is 0.116.
- **`no_size`** drops area, duration, severity, and the five weather columns summed over the event window. It keeps what could be known before the flood is mapped.

In [7]:
for name, spec in impact.VARIANTS.items():
    print(name, "| numeric:", spec["numeric"], "| categorical:", spec["categorical"])
print("std of log10(severity):", round(float(np.log10(frame["Severity"]).std()), 3))

brief | numeric: ['Log_Area', 'Event_Duration_Days', 'Total_Precipitation_mm', 'Max_24h_Precipitation_mm', 'Total_Liquid_Rain_mm', 'Snow_Melt_Equiv_mm', 'Antecedent_7Day_Precip_mm', 'Estimated_Surface_Runoff_mm', 'Mean_Temp_2m_C', 'Max_Temp_2m_C', 'Min_Temp_2m_C', 'Max_Wind_Gust_10m_kmh', 'Min_Sea_Level_Pressure_hPa', 'Mean_Relative_Humidity_pct', 'Topsoil_Moisture_0_7cm_m3m3', 'Cumulative_ET0_Evapotranspiration_mm'] | categorical: ['Koppen_Climate_Zone', 'Continent_Region']
no_size | numeric: ['Max_24h_Precipitation_mm', 'Antecedent_7Day_Precip_mm', 'Mean_Temp_2m_C', 'Max_Temp_2m_C', 'Min_Temp_2m_C', 'Max_Wind_Gust_10m_kmh', 'Min_Sea_Level_Pressure_hPa', 'Mean_Relative_Humidity_pct', 'Topsoil_Moisture_0_7cm_m3m3', 'Latitude', 'Longitude', 'Start_Month'] | categorical: ['Koppen_Climate_Zone', 'Continent_Region', 'Cause_Group']
std of log10(severity): 0.116


## 5. Temporal split

| Split | Start years | Role |
|---|---|---|
| fit | 1985–2015 | first fit |
| valid | 2016–2018 | picks the number of trees (early stopping, `use_best_model`) |
| train | 1985–2018 | refit with that tree count; this model is scored |
| test | 2019–2024 | scored once |

The served `.cbm` files are refit on all 5,503 rows with the same tree count.

In [8]:
masks = impact.split_masks(frame)
pd.Series({name: int(mask.sum()) for name, mask in masks.items()}).to_frame("events")

,events
fit,4316
valid,395
train,4711
test,792


## 6. Train the brief variant

`fit_variant()` runs the three steps above. `final=False` skips the last refit on every row, which is only needed for the saved model.

In [9]:
brief = impact.fit_variant(frame, "brief", final=False)
print("trees:", brief["iterations"])
print("validation RMSE, first tree:", brief["validation_rmse_first_iteration"], " best:", brief["validation_rmse_best"])
print("train RMSE:", brief["train_rmse"])
print("temporal test:", brief["temporal_test"])
pd.Series(brief["feature_importance"]).head(8).to_frame("importance (sums to 100)")

trees: 499
validation RMSE, first tree: 0.8511  best: 0.0922
train RMSE: 0.0859
temporal test: {'rmse': 0.162, 'mae': 0.1001, 'r2': 0.9824, 'n': 792}


,importance (sums to 100)
Log_Area,73.335
Event_Duration_Days,9.996
Cumulative_ET0_Evapotranspiration_mm,5.585
Total_Precipitation_mm,3.128
Total_Liquid_Rain_mm,2.254
Estimated_Surface_Runoff_mm,1.826
Topsoil_Moisture_0_7cm_m3m3,0.806
Max_24h_Precipitation_mm,0.690


## 7. Baselines on the same 792 test events

`baselines()` scores four references on the 2019–2024 rows: the formula, the formula without severity, a linear regression on log area and log days, and the training mean.

In [10]:
no_size = impact.fit_variant(frame, "no_size", final=False)
rows = {name: scores for name, scores in impact.baselines(frame).items()}
rows["catboost_brief"] = brief["temporal_test"]
rows["catboost_no_size"] = no_size["temporal_test"]
table = pd.DataFrame(rows).T[["rmse", "mae", "r2", "n"]].sort_values("rmse")
table

,rmse,mae,r2,n
dfo_formula,0.0292,0.0253,0.9994,792.0
linear_log_area_log_days,0.1136,0.0949,0.9913,792
dfo_formula_without_severity,0.1591,0.1245,0.983,792.0
catboost_brief,0.162,0.1001,0.9824,792.0
catboost_no_size,0.8994,0.6833,0.4559,792.0
train_mean,1.2327,0.959,-0.0222,792.0


The formula is exact up to rounding. The linear regression learns coefficients close to 1 on both logs, so it is the formula with severity folded into the intercept. CatBoost does worse because trees fit a straight line in steps and flatten at the ends of the range. On the test rows it over-predicts events with an index of 4 or less by 0.11 on average and under-predicts those between 6 and 7 by 0.07. Leaving severity out adds a bias of its own: +0.08 on severity-1 events and −0.12 on severity-2 events.

In [11]:
impact.baselines(frame)["linear_log_area_log_days"]["coefficients"]

{'log_area': 1.0164, 'log_days': 1.0399, 'intercept': -0.0343}

## 8. The brief's own protocol

The brief reports test RMSE 0.0985 at iteration 1088. That matches a random 80/20 split where the best iteration is chosen on the test rows themselves. Choosing it that way is optimistic. The same protocol here:

In [12]:
pd.Series(impact.brief_reproduction(frame)).to_frame("value")

,value
protocol,"random 80/20 split (random_state 42), use_best..."
caveat,Choosing the iteration on the rows it is score...
best_iteration,1195
test_rmse_first_iteration,1.0813
test_rmse_best,0.1038
train_rmse_last_iteration,0.0754
dfo_formula_rmse_same_rows,0.0287
brief_reported,"{'test_rmse': 0.0985, 'train_rmse': 0.0695, 'b..."


## 9. The no_size variant

Without the formula inputs the model explains about 45% of the variance. Its top inputs are the 24-hour maximum rain and topsoil moisture. Both are synthetic, and both correlate with event length in this file. So this skill measures how the enrichment was generated, not how floods behave.

In [13]:
print("trees:", no_size["iterations"], " test:", no_size["temporal_test"])
pd.Series(no_size["mean_abs_shap_test"]).head(8).to_frame("mean |SHAP| on test")

trees: 273  test: {'rmse': 0.8994, 'mae': 0.6833, 'r2': 0.4559, 'n': 792}


,mean |SHAP| on test
Max_24h_Precipitation_mm,0.3163
Topsoil_Moisture_0_7cm_m3m3,0.2876
Cause_Group,0.0768
Antecedent_7Day_Precip_mm,0.0714
Continent_Region,0.0705
Longitude,0.0374
Latitude,0.0366
Mean_Temp_2m_C,0.0291


## 10. One event through the API layer

`api/app/impact.py` loads the committed `.cbm` files and the metrics card. It turns an API request into the model's columns and returns the estimate with CatBoost SHAP values. The SHAP values add up to the estimate. When area, dates and severity are sent, it also returns the formula value.

Event 5477 is the Kenya flood of October–November 2023, a test-period event. The served model was refit on every row, so it has seen this event.

In [14]:
from api.app import impact as api_impact

row = frame[frame.Event_ID == 5477].iloc[0]
event = api_impact.ImpactEventIn(
    start_date=row.Start_Date, end_date=row.End_Date, area_km2=row.Area_km2, severity=row.Severity,
    latitude=row.Latitude, longitude=row.Longitude, koppen_climate_zone=row.Koppen_Climate_Zone,
    continent_region=row.Continent_Region, main_cause=row.Main_Cause,
    **{field: row[column] for field, column in api_impact.WEATHER_FIELDS.items()},
)
for variant in ("brief", "no_size"):
    out = api_impact.predict([event], variant)[0]
    print(variant, "| estimate", out["flood_impact_index"], "| formula", out["dfo_formula"]["value"],
          "| recorded", row[impact.TARGET], "| SHAP sum check",
          round(out["expected_value"] + sum(c["contribution"] for c in out["contributions"]), 3))
pd.DataFrame(api_impact.predict([event], "brief")[0]["contributions"]).head(6)

brief | estimate 7.027 | formula 6.985 | recorded 7.0 | SHAP sum check 7.027
no_size | estimate 6.885 | formula 6.985 | recorded 7.0 | SHAP sum check 6.885


,feature,input,contribution
0,Log_Area,5.361416,0.7094
1,Event_Duration_Days,27,0.2863
2,Cumulative_ET0_Evapotranspiration_mm,145.7,0.2057
3,Total_Precipitation_mm,793.3,0.1343
4,Estimated_Surface_Runoff_mm,502.0,0.1290
5,Total_Liquid_Rain_mm,793.3,0.1205


## 11. Rebuild the artifacts

The notebook does not write files. To refit and save both models, the metrics card and the held-out predictions:

```bash
python3 -m model.impact            # from the committed event table
python3 -m model.impact --rebuild  # rebuild the event table from data/ first
pytest tests/test_impact_model.py tests/test_impact_api.py
```